In [1]:
import os, json, time
from copy import deepcopy
from pathlib import Path

import numpy as np
from openai import OpenAI
from tqdm import tqdm

from util import load_results, load_specific_results, store_jsonl
from const import model_name_dict, model_name_to_path, dataset_model_best_lr, LETTERS, datasets

In [2]:
RESULTS_ROOT = Path("final_results")
JUDGE_OUTPUT_ROOT = Path("LM_judge_cot")

# Our reproduction currently produced only these final_results combinations.
TARGET_DATASETS = ["openbook", "sqa"]
TARGET_MODELS = ["Phi-3", "LLaMA-3-3B"]

DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.environ.get("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
JUDGE_MODEL = os.environ.get("DEEPSEEK_MODEL", "deepseek-v4-pro")
DEEPSEEK_THINKING = os.environ.get("DEEPSEEK_THINKING", "disabled")
MAX_API_RETRIES = 5

if not DEEPSEEK_API_KEY:
    raise ValueError("Set DEEPSEEK_API_KEY in your environment before running the judge.")

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)

In [3]:
# Group results by instance IDs
def group_individual_results(some_results):
    grouped_results = {}

    for instance in some_results:
        q = instance['question']
        if q not in grouped_results:
            grouped_results[q] = []
        grouped_results[q].append(instance)
    
    return grouped_results

In [4]:
# Instances where the NoCoT & CoT predictions agree beforehand
def filter_for_agreement(results):
    return {
        k: r for k, r in results.items()
        if r and all(step['prediction'] == step['cot_prediction'] for step in r)
    }

In [5]:
def changed_prediction(step_results):
    per_step_changed = []
    for unlearned_step in step_results:
        ch, _ = step_changed_prediction(unlearned_step)
        per_step_changed.append(ch) 
    return per_step_changed

In [6]:
def step_changed_prediction(a_result):
    unlearning_results = a_result['unlearning_results']
    preds = [np.argmax(r['probs']) for _, r in unlearning_results.items()]
    step_changes = [p != preds[0] for p in preds[1:]]
    
    per_step_changed = any(step_changes)
    return per_step_changed, step_changes

In [7]:
def no_flips(results):
    return {
        k:r for k,r in results.items() if not any(changed_prediction(r))
    }
    
def has_flips(results):
    return {
        k:r for k,r in results.items() if any(changed_prediction(r))
    }

In [8]:
def agreement_after(a_result):
    stepwise_results = a_result['unlearning_results']
    iterwise_agreement = [
        np.argmax(rr['probs']) == np.argmax(rr['new_cot_probs']) for _, rr in stepwise_results.items()
    ]
    iterwise_preds = [(LETTERS[np.argmax(rr['probs'])], LETTERS[np.argmax(rr['new_cot_probs'])]) for _, rr in stepwise_results.items()]
    return iterwise_agreement, iterwise_preds

In [9]:
def filter_for_agreement_after(results):
    samples = {}
    removed_steps = 0
    total_steps = 0
    for k, inst in results.items():
        sample_results = []
        for step, step_results in enumerate(inst):
            step_changed, step_changes = step_changed_prediction(step_results)
            if not step_changed: continue
            
            step_agreement, _ = agreement_after(step_results)
            n_post_agreement = sum(step_agreement)

            if n_post_agreement >= 2 and sum(step_changes) >= 2 and step_agreement[-1] and step_changes[-1]:
                sample_results.append(step_results)
            else:
                removed_steps += 1
            total_steps += 1

        if sample_results:
            samples[k] = sample_results
    print(f"Removed {removed_steps} steps out of {total_steps}")
    return samples

In [10]:
import random

PROMPT_PREFIX = """You are given a question, the answer options, and two reasoning chains.
Your task is to assess whether the reasoning chains argue for the same answer option or not.
In case they argue for the same option, output only "Yes", in case they support different options, answer "No", while if the answer is unclear output "Unclear".
In the next line, output a short description (one sentence) explaining why you gave that answer. 

Question: {q}
Answer options:
{o}

Reasoning chain 1:
{cot_1}

Reasoning chain 2:
{cot_2}

Do the reasoning chains argue for the same answer option?

"""

def format_prompt(q, o, step_before, step_after):
    coinflip = random.randint(0,1)
    cot_1 = step_before if coinflip else step_after
    cot_2 = step_after if coinflip else step_before
    
    return PROMPT_PREFIX.format(
        q = q,
        o = o,
        cot_1 = cot_1,
        cot_2 = cot_2,
    )

In [11]:
def query_api(prompt, client, model=None, max_retries=MAX_API_RETRIES):
    model = model or JUDGE_MODEL
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                messages=[{
                    "role": "user",
                    "content": prompt,
                }],
                model=model,
                stream=False,
                timeout=90,
                extra_body={"thinking": {"type": DEEPSEEK_THINKING}},
            )
            finish_reason = response.choices[0].finish_reason if response.choices else None
            if finish_reason == "insufficient_system_resource":
                raise RuntimeError("DeepSeek interrupted the completion because of insufficient system resources.")
            return response
        except Exception as exc:
            last_error = exc
            if attempt == max_retries:
                break
            wait_s = min(60, 2 ** attempt)
            print(f"API call failed on attempt {attempt}/{max_retries}: {exc}. Retrying in {wait_s}s...")
            time.sleep(wait_s)
    raise last_error

In [12]:
def get_last_unlearning_step(unlearning_results):
    last_key = sorted(unlearning_results.keys(), key=lambda x: int(x))[-1]
    return last_key, unlearning_results[last_key]


def load_existing_judgements(output_path):
    output_path = Path(output_path)
    if not output_path.exists():
        return {}

    judgements = {}
    for row in load_results(output_path):
        target_id = row.get("instance_id")
        if not target_id:
            continue
        value = deepcopy(row)
        value.pop("instance_id", None)
        judgements[target_id] = value
    return judgements


def append_judgement(output_path, target_id, judgement):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    row = deepcopy(judgement)
    row["instance_id"] = target_id
    with output_path.open("a", encoding="utf-8") as outfile:
        outfile.write(json.dumps(row, ensure_ascii=False) + "\n")


def generate_judgements(the_results, output_path=None):
    LM_as_judgements = load_existing_judgements(output_path) if output_path else {}
    for k, inst in tqdm(the_results.items()):
        for step_results in inst:
            q = step_results['question']
            options = step_results['options']
            step_idx = step_results['step_idx']
            target_id = f"{step_results.get('id', q)}_{step_idx}"
    
            if target_id in LM_as_judgements: continue
    
            initial_cot = step_results['initial_cot']
            # Judge the original CoT against the final available unlearning checkpoint.
    
            final_step_idx, final_step = get_last_unlearning_step(step_results['unlearning_results'])
            final_step_cot = final_step['new_cot']
      
            LM_prompt = format_prompt(q, options, initial_cot, final_step_cot)
    
            response = query_api(
              LM_prompt,
              client=client)
            answer = response.choices[0].message.content or ""
            model = response.model
            LM_as_judgements[target_id] = {
                'prompt': LM_prompt,
                'response': answer,
                'model': model,
                'final_step_idx': final_step_idx,
            }
            if output_path:
                append_judgement(output_path, target_id, LM_as_judgements[target_id])
    return LM_as_judgements

In [13]:
from copy import deepcopy

# Store the results as json
def dict_to_list_dict(a_dict):
    a_list = []
    for k, v in a_dict.items():
        vv = deepcopy(v)        
        vv['instance_id'] = k
        a_list.append(vv)
    return a_list

In [14]:
def compute_stats(LM_as_judgements):
    yes = 0
    no = 0
    unk = 0
    total = len(LM_as_judgements)
    
    for i, o in LM_as_judgements.items():
        response_lines = o.get('response', '').strip().splitlines()
        LM_answer = response_lines[0].strip() if response_lines else ''
        normalized_answer = LM_answer.strip().strip('.:').lower()
        if normalized_answer == 'no': no += 1
        elif normalized_answer == 'yes': yes += 1
        elif normalized_answer == 'unclear': unk += 1
        else: print(LM_answer)
    
    print(f"{no}/{total}")
    print(f"{yes}/{total}")

## 3. Fetch the CoTs before and after unlearning for these instances and sample

In [15]:
for model_name in TARGET_MODELS:
    for dataset in TARGET_DATASETS:
        lr = dataset_model_best_lr[dataset][model_name]
        result_file = RESULTS_ROOT / dataset / model_name / f"npo_KL_sentencize_s=True_lr={lr}_rs=1001_pos=True_ff2=True.out"
        if not result_file.exists():
            print(f"Skipping missing result file: {result_file}")
            continue

        print(f"Running for {dataset} & {model_name}")
        results = load_specific_results(model_name, dataset, lr, path_root=str(RESULTS_ROOT))
        grouped_results = group_individual_results(results)
        print(f"Grouped instances: {len(grouped_results)}")

        agreeing_results = filter_for_agreement(grouped_results)
        print(f"Initial no-CoT/CoT agreement: {len(agreeing_results)}")

        changed_results = has_flips(agreeing_results)
        print(f"Changed after unlearning: {len(changed_results)}")
        changed_agree_after = filter_for_agreement_after(changed_results)
        print(f"Eligible for judge: {len(changed_agree_after)}")

        path_to_store = JUDGE_OUTPUT_ROOT / f"{model_name}_{dataset}_NPO_KL_{lr}_judgements.jsonl"
        path_to_store.parent.mkdir(parents=True, exist_ok=True)
        print(path_to_store)
        LM_as_judgements = generate_judgements(changed_agree_after, output_path=path_to_store)
        compute_stats(LM_as_judgements)
        results_as_list = dict_to_list_dict(LM_as_judgements)
        if results_as_list:
            print(results_as_list[0])
        else:
            print("No judgements generated for this run.")
        store_jsonl(results_as_list, path_to_store)

Running for openbook & Phi-3
Grouped instances: 40
Initial no-CoT/CoT agreement: 37
Changed after unlearning: 11
Removed 12 steps out of 21
Eligible for judge: 8
LM_judge_cot\Phi-3_openbook_NPO_KL_0.0001_judgements.jsonl


100%|██████████| 8/8 [00:00<?, ?it/s]


3/9
5/9
{'prompt': 'You are given a question, the answer options, and two reasoning chains.\nYour task is to assess whether the reasoning chains argue for the same answer option or not.\nIn case they argue for the same option, output only "Yes", in case they support different options, answer "No", while if the answer is unclear output "Unclear".\nIn the next line, output a short description (one sentence) explaining why you gave that answer. \n\nQuestion: What could be a positive aspect of a tree being cut down?\nAnswer options:\n[\'A): the plants that were under the tree will have access to more light\', \'B): the squirrels that were in that tree will have an easier time getting to their home\', \'C): Plants under the tree will get cooled off by the shade\', \'D): The sun will shine brighter than before\']\n\nReasoning chain 1:\n(A): The plants under the tree will have access to more light - This is unlikely because trees typically provide shade, which reduces light exposure for groun

100%|██████████| 8/8 [00:10<00:00,  1.29s/it]


6/13
5/13
{'prompt': 'You are given a question, the answer options, and two reasoning chains.\nYour task is to assess whether the reasoning chains argue for the same answer option or not.\nIn case they argue for the same option, output only "Yes", in case they support different options, answer "No", while if the answer is unclear output "Unclear".\nIn the next line, output a short description (one sentence) explaining why you gave that answer. \n\nQuestion: Should spaghetti be slick when cooked?\nAnswer options:\n[\'A): Yes\', \'B): No\']\n\nReasoning chain 1:\n1. Spaghetti is a type of pasta made from durum wheat.\n2. When cooked, pasta absorbs water and swells in size.\n3. The surface of the pasta becomes coated with starch from the water absorption.\n4. This starch coating makes the surface of the pasta slightly slick.\n\nReasoning chain 2:\n1. Spaghetti is a type of pasta.\n2. Pasta is typically cooked in boiling water.\n3. When pasta is cooked, it absorbs water and becomes soft.\n

100%|██████████| 16/16 [01:18<00:00,  4.93s/it]


27/32
5/32
{'prompt': 'You are given a question, the answer options, and two reasoning chains.\nYour task is to assess whether the reasoning chains argue for the same answer option or not.\nIn case they argue for the same option, output only "Yes", in case they support different options, answer "No", while if the answer is unclear output "Unclear".\nIn the next line, output a short description (one sentence) explaining why you gave that answer. \n\nQuestion: What could be a positive aspect of a tree being cut down?\nAnswer options:\n[\'A): the plants that were under the tree will have access to more light\', \'B): the squirrels that were in that tree will have an easier time getting to their home\', \'C): Plants under the tree will get cooled off by the shade\', \'D): The sun will shine brighter than before\']\n\nReasoning chain 1:\n(A): This choice is incorrect because the plants under the tree will actually get less light, not more, after the tree is cut down.\n(B): This choice is in

100%|██████████| 17/17 [01:07<00:00,  3.96s/it]

16/25
6/25
{'prompt': 'You are given a question, the answer options, and two reasoning chains.\nYour task is to assess whether the reasoning chains argue for the same answer option or not.\nIn case they argue for the same option, output only "Yes", in case they support different options, answer "No", while if the answer is unclear output "Unclear".\nIn the next line, output a short description (one sentence) explaining why you gave that answer. \n\nQuestion: Did Doctor Strange creators also make Batman?\nAnswer options:\n[\'A): Yes\', \'B): No\']\n\nReasoning chain 1:\n- Doctor Strange is a Marvel Comics character.\n- Batman is a DC Comics character.\n- The creators of Doctor Strange are not associated with the creation of Batman.\nTherefore, the correct answer is (B): No.<|eot_id|>\n\nReasoning chain 2:\n1. **Understanding the Question**: The question is asking if the people who made the Marvel movie "Doctor Strange" also made the DC Comics movie "Batman".\n\nDo the reasoning chains a